# 🛒 Superstore Sales — Data Cleaning & Transformation
**Project:** Sales Analysis of a Superstore  
**Goal:** Load raw CSV data, clean it, engineer features, and export a analysis-ready dataset.  
**Output:** `superstore_sales_clean.csv`

---


## 0. Install & Import Libraries

In [ ]:
# Install any missing libraries (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


## 1. Load Raw Data

In [ ]:
df = pd.read_csv('superstore_sales_data.csv', parse_dates=['Order_Date', 'Ship_Date'])

print(f"Shape : {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


In [ ]:
# Data types overview
df.info()


## 2. Initial Exploratory Data Analysis

In [ ]:
# Basic statistics for numeric columns
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std', 'min', 'max'])


In [ ]:
# Unique value counts for categorical columns
cat_cols = ['Category', 'Sub_Category', 'Region', 'Segment', 'Ship_Mode', 'State']

for col in cat_cols:
    print(f"{col:15s} → {df[col].nunique():3d} unique values : {sorted(df[col].unique())}")


## 3. Handle Missing Values

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

if missing_df.empty:
    print("✅ No missing values found!")
else:
    print(missing_df)


In [ ]:
# Visualise missing data heatmap
plt.figure(figsize=(14, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Detect & Remove Duplicates

In [ ]:
dupes = df.duplicated()
print(f"Duplicate rows found : {dupes.sum()}")

if dupes.sum() > 0:
    print("\nDuplicate rows:")
    display(df[dupes])
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"\n✅ Duplicates removed. New shape: {df.shape}")
else:
    print("✅ No duplicates found.")


## 5. Fix Data Types

In [ ]:
# Ensure date columns are datetime
for col in ['Order_Date', 'Ship_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Ensure numeric columns are correct types
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Standardise string columns (strip whitespace, title-case names)
str_cols = ['Category', 'Sub_Category', 'Region', 'Segment', 'Ship_Mode',
            'State', 'City', 'Country', 'Customer_Name', 'Product_Name']
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

# Standardise Category & Region to Title Case
for col in ['Category', 'Sub_Category', 'Region', 'Segment', 'Ship_Mode']:
    df[col] = df[col].str.title()

print("✅ Data types corrected.")
df.dtypes


## 6. Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Outlier Detection — Box Plots', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flatten(), ['Sales', 'Profit', 'Quantity', 'Discount']):
    df.boxplot(column=col, ax=ax, color='steelblue')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()


In [ ]:
# IQR-based outlier summary (flag, not drop — negative profit is valid)
def iqr_outlier_info(series, col_name):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"{col_name:10s} | Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}  "
          f"Bounds=[{lower:.2f}, {upper:.2f}]  Outliers={len(outliers)}")

print(f"{'Column':10s} | Summary")
print("-" * 70)
for col in ['Sales', 'Profit', 'Quantity', 'Discount']:
    iqr_outlier_info(df[col], col)


## 7. Business Rule Validation

In [ ]:
issues = {}

# Rule 1: Sales must be positive
neg_sales = df[df['Sales'] <= 0]
issues['Negative/Zero Sales'] = len(neg_sales)

# Rule 2: Discount must be between 0 and 1
bad_discount = df[~df['Discount'].between(0, 1)]
issues['Invalid Discount (not 0–1)'] = len(bad_discount)

# Rule 3: Quantity must be >= 1
bad_qty = df[df['Quantity'] < 1]
issues['Quantity < 1'] = len(bad_qty)

# Rule 4: Ship date must not be before order date
bad_dates = df[df['Ship_Date'] < df['Order_Date']]
issues['Ship Date before Order Date'] = len(bad_dates)

# Rule 5: Order IDs must not be empty
empty_ids = df[df['Order_ID'].isnull() | (df['Order_ID'] == '')]
issues['Empty Order IDs'] = len(empty_ids)

print("Business Rule Validation Results:")
print("-" * 40)
all_ok = True
for rule, count in issues.items():
    status = "✅" if count == 0 else "⚠️"
    if count > 0:
        all_ok = False
    print(f"  {status}  {rule}: {count} violations")

if all_ok:
    print("\n✅ All business rules passed!")


## 8. Feature Engineering

In [ ]:
# ── Temporal features ──────────────────────────────────────────────────────
df['Order_Year']       = df['Order_Date'].dt.year
df['Order_Month']      = df['Order_Date'].dt.month
df['Order_Month_Name'] = df['Order_Date'].dt.strftime('%B')
df['Order_Quarter']    = df['Order_Date'].dt.quarter.map(
                             {1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
df['Order_DayOfWeek']  = df['Order_Date'].dt.day_name()

# ── Shipping features ──────────────────────────────────────────────────────
df['Shipping_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days

# ── Financial features ─────────────────────────────────────────────────────
df['Profit_Margin_Pct'] = (df['Profit'] / df['Sales'].replace(0, np.nan) * 100).round(2)
df['Revenue_After_Discount'] = (df['Sales'] * (1 - df['Discount'])).round(2)
df['Profit_Per_Unit'] = (df['Profit'] / df['Quantity'].replace(0, np.nan)).round(2)

# ── Profit flag ────────────────────────────────────────────────────────────
df['Profit_Flag'] = np.where(df['Profit'] > 0, 'Profitable',
                    np.where(df['Profit'] < 0, 'Loss', 'Break-Even'))

# ── Order size segment ─────────────────────────────────────────────────────
df['Order_Size'] = pd.cut(
    df['Sales'],
    bins=[0, 100, 500, 1000, np.inf],
    labels=['Small (<$100)', 'Medium ($100–$500)', 'Large ($500–$1K)', 'Enterprise (>$1K)']
)

# ── Discount band ──────────────────────────────────────────────────────────
df['Discount_Band'] = pd.cut(
    df['Discount'],
    bins=[-0.001, 0, 0.2, 0.4, 1.0],
    labels=['No Discount', 'Low (1–20%)', 'Medium (21–40%)', 'High (>40%)']
)

print(f"✅ Feature engineering complete. New shape: {df.shape}")
print(f"
New columns added:")
new_cols = ['Order_Year','Order_Month','Order_Month_Name','Order_Quarter',
            'Order_DayOfWeek','Shipping_Days','Profit_Margin_Pct',
            'Revenue_After_Discount','Profit_Per_Unit',
            'Profit_Flag','Order_Size','Discount_Band']
for c in new_cols:
    print(f"  • {c}")


## 9. Post-Cleaning Summary

In [ ]:
print("=" * 55)
print("       CLEAN DATASET SUMMARY")
print("=" * 55)
print(f"  Total Records       : {len(df)}")
print(f"  Total Columns       : {df.shape[1]}")
print(f"  Date Range          : {df['Order_Date'].min().date()} → {df['Order_Date'].max().date()}")
print(f"  Total Sales         : ${df['Sales'].sum():,.2f}")
print(f"  Total Profit        : ${df['Profit'].sum():,.2f}")
print(f"  Overall Margin      : {df['Profit'].sum()/df['Sales'].sum()*100:.2f}%")
print(f"  Categories          : {', '.join(df['Category'].unique())}")
print(f"  Regions             : {', '.join(sorted(df['Region'].unique()))}")
print(f"  Sub-Categories      : {df['Sub_Category'].nunique()}")
print(f"  Profitable Orders   : {(df['Profit_Flag']=='Profitable').sum()}")
print(f"  Loss Orders         : {(df['Profit_Flag']=='Loss').sum()}")
print("=" * 55)


## 10. Exploratory Visualisations on Clean Data

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Superstore Sales — Clean Data Overview', fontsize=16, fontweight='bold', y=1.01)

palette = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63', '#9C27B0']

# ── 1. Sales by Category ─────────────────────────────────────────────────
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
axes[0,0].bar(cat_sales.index, cat_sales.values, color=palette[:3], edgecolor='white')
axes[0,0].set_title('Total Sales by Category', fontweight='bold')
axes[0,0].set_ylabel('Sales ($)')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(cat_sales.values):
    axes[0,0].text(i, v + 500, f'${v:,.0f}', ha='center', fontsize=9)

# ── 2. Profit by Category ────────────────────────────────────────────────
cat_profit = df.groupby('Category')['Profit'].sum().sort_values(ascending=False)
colors_p = ['#4CAF50' if v >= 0 else '#F44336' for v in cat_profit.values]
axes[0,1].bar(cat_profit.index, cat_profit.values, color=colors_p, edgecolor='white')
axes[0,1].set_title('Total Profit by Category', fontweight='bold')
axes[0,1].set_ylabel('Profit ($)')
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0,1].axhline(0, color='black', linewidth=0.8, linestyle='--')

# ── 3. Monthly Sales Trend ────────────────────────────────────────────────
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly = df.groupby('Order_Month_Name')[['Sales','Profit']].sum()
monthly = monthly.reindex(month_order).fillna(0)
axes[0,2].plot(range(len(monthly)), monthly['Sales'],   marker='o', color='#2196F3', label='Sales',  linewidth=2)
axes[0,2].plot(range(len(monthly)), monthly['Profit'],  marker='s', color='#FF9800', label='Profit', linewidth=2)
axes[0,2].set_xticks(range(len(monthly)))
axes[0,2].set_xticklabels([m[:3] for m in month_order], rotation=45)
axes[0,2].set_title('Monthly Sales & Profit Trend', fontweight='bold')
axes[0,2].legend()
axes[0,2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── 4. Top 5 Sub-Categories ───────────────────────────────────────────────
top5 = df.groupby('Sub_Category')['Sales'].sum().nlargest(5)
axes[1,0].barh(top5.index, top5.values, color='#2196F3', edgecolor='white')
axes[1,0].set_title('Top 5 Sub-Categories by Sales', fontweight='bold')
axes[1,0].set_xlabel('Sales ($)')
axes[1,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1,0].invert_yaxis()

# ── 5. Profit Flag Distribution ───────────────────────────────────────────
flag_counts = df['Profit_Flag'].value_counts()
flag_colors = {'Profitable':'#4CAF50','Loss':'#F44336','Break-Even':'#FFC107'}
axes[1,1].pie(flag_counts.values,
              labels=flag_counts.index,
              autopct='%1.1f%%',
              colors=[flag_colors.get(l,'grey') for l in flag_counts.index],
              startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1,1].set_title('Profit vs Loss Orders', fontweight='bold')

# ── 6. Sales by Region ────────────────────────────────────────────────────
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
axes[1,2].bar(reg_sales.index, reg_sales.values, color=palette[:4], edgecolor='white')
axes[1,2].set_title('Total Sales by Region', fontweight='bold')
axes[1,2].set_ylabel('Sales ($)')
axes[1,2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(reg_sales.values):
    axes[1,2].text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('superstore_eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Charts saved to superstore_eda_charts.png")


In [ ]:
# Correlation heatmap for numeric features
num_features = ['Sales', 'Quantity', 'Discount', 'Profit',
                'Shipping_Days', 'Profit_Margin_Pct', 'Revenue_After_Discount']

corr = df[num_features].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('superstore_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Correlation heatmap saved.")


In [ ]:
plt.figure(figsize=(10, 5))
colors_scatter = df['Profit_Flag'].map({'Profitable':'#4CAF50','Loss':'#F44336','Break-Even':'#FFC107'})
plt.scatter(df['Discount'], df['Profit'], c=colors_scatter, alpha=0.6, edgecolors='white', linewidth=0.4)
plt.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
plt.xlabel('Discount Rate', fontsize=12)
plt.ylabel('Profit ($)', fontsize=12)
plt.title('Discount Rate vs Profit — Scatter Plot', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4CAF50', label='Profitable'),
                   Patch(facecolor='#F44336', label='Loss'),
                   Patch(facecolor='#FFC107', label='Break-Even')]
plt.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

# Quick insight
high_discount_loss = df[(df['Discount'] >= 0.4) & (df['Profit'] < 0)]
print(f"⚠️  Orders with ≥40% discount AND negative profit: {len(high_discount_loss)}")


## 11. Export Clean Dataset

In [ ]:
# Reorder columns logically
col_order = [
    # Identifiers
    'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode',
    # Temporal
    'Order_Year', 'Order_Month', 'Order_Month_Name', 'Order_Quarter', 'Order_DayOfWeek',
    'Shipping_Days',
    # Customer
    'Customer_ID', 'Customer_Name', 'Segment',
    # Geography
    'Country', 'City', 'State', 'Region',
    # Product
    'Product_ID', 'Category', 'Sub_Category', 'Product_Name',
    # Financials (original)
    'Sales', 'Quantity', 'Discount', 'Profit',
    # Derived financials
    'Revenue_After_Discount', 'Profit_Margin_Pct', 'Profit_Per_Unit',
    # Categorical flags
    'Profit_Flag', 'Order_Size', 'Discount_Band'
]

df_clean = df[col_order].copy()

output_path = 'superstore_sales_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"✅ Clean dataset exported → {output_path}")
print(f"   Rows    : {df_clean.shape[0]}")
print(f"   Columns : {df_clean.shape[1]}")
print(f"
Column list:")
for i, c in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {c}")


---
## ✅ Pipeline Complete

| Step | Action |
|------|--------|
| 1 | Loaded raw CSV with correct dtypes |
| 2 | Explored shape, dtypes, and value distributions |
| 3 | Checked & confirmed no missing values |
| 4 | Detected and removed duplicates |
| 5 | Standardised string and date columns |
| 6 | Detected outliers via IQR method |
| 7 | Validated business rules (sales, discounts, dates) |
| 8 | Engineered 12 new features (temporal, financial, categorical) |
| 9 | Generated EDA visualisations |
| 10 | Exported `superstore_sales_clean.csv` |

The clean dataset is ready for Power BI, Tableau, or further ML modelling.
